In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from textblob import TextBlob
import nltk
nltk.download('brown', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

True

In [3]:
!pip install sumy
!pip install sentence_splitter

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 106.8 MB/s eta 0:00:00
  Created wheel for breadability: filename=breadability-0.1.20-py2.py3-none-any.whl size=21695 sha256=bbd9957ac026619e8b0580a0334b3b1b930b6df4738cddb72c63157cd43b9c11
  Stored in directory: /root/.cache/pip/wheels/32/99/64/59305409cacd03aa03e7bddf31a9db34b1fa7033bd41972662
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=dea2d73a54c66fd5d74dfc41be73737c256b4e99e96f82d438f7ac2e408bf56c
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built breadability docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.3 MB/s eta 0:00:00


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import random
import os
from collections import Counter
from typing import List, Optional, Union, Dict, Any
from abc import ABC, abstractmethod
from tqdm import tqdm


# Base Abstract Class
class BaseTranslator(ABC):
    """Abstract base class for all translation models."""

    def __init__(self, device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.is_loaded = False

    @abstractmethod
    def load_model(self, model_path: str):
        """Load the translation model."""
        pass

    @abstractmethod
    def translate_text(self, text: str, **kwargs) -> str:
        """Translate a single text."""
        pass

    @abstractmethod
    def translate_batch(self, texts: List[str], **kwargs) -> List[str]:
        """Translate a batch of texts."""
        pass

    def get_model_info(self) -> dict:
        """Get basic model information."""
        return {
            "model_type": self.__class__.__name__,
            "device": self.device,
            "is_loaded": self.is_loaded
        }


# LSTM Model Components
class Vocabulary:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = 1

    def __len__(self):
        return len(self.stoi)

    @staticmethod
    def tokenizer(text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers,
                            dropout=dropout, batch_first=True,
                            bidirectional=True)

        self.dropout = nn.Dropout(dropout)
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)

        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        cell_cat = torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)

        hidden = torch.tanh(self.fc_hidden(hidden_cat))
        cell = torch.tanh(self.fc_cell(cell_cat))

        return hidden.unsqueeze(0), cell.unsqueeze(0)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = False

        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs


# Individual Translator Classes
class LSTMTranslator(BaseTranslator):
    """Custom LSTM-based translator."""

    def __init__(self, device: Optional[str] = None, config: Optional[Dict] = None):
        super().__init__(device)
        self.config = config or self._get_default_config()
        self.src_vocab = None
        self.trg_vocab = None

    def _get_default_config(self):
        return {
            'ENC_EMB_DIM': 300,
            'DEC_EMB_DIM': 300,
            'HID_DIM': 256,
            'N_LAYERS': 1,
            'ENC_DROPOUT': 0.5,
            'DEC_DROPOUT': 0.5,
            'vocab_data_url': "https://drive.google.com/uc?id=12l_YUBjcAT3CnEYCqmMQwFwTj9ft5N7M"
        }

    def load_model(self, model_path: str):
        """Load LSTM model and build vocabularies."""
        try:
            self.src_vocab = Vocabulary()
            self.trg_vocab = Vocabulary()

            # Load training data for vocabulary building
            train_df = pd.read_csv(self.config['vocab_data_url'])
            self.src_vocab.build_vocabulary(train_df['text'].tolist())
            self.trg_vocab.build_vocabulary(train_df['english_translation'].tolist())

            # Initialize model architecture
            encoder = Encoder(
                len(self.src_vocab),
                self.config['ENC_EMB_DIM'],
                self.config['HID_DIM'],
                self.config['N_LAYERS'],
                self.config['ENC_DROPOUT']
            )
            decoder = Decoder(
                len(self.trg_vocab),
                self.config['DEC_EMB_DIM'],
                self.config['HID_DIM'],
                self.config['N_LAYERS'],
                self.config['DEC_DROPOUT']
            )
            self.model = Seq2Seq(encoder, decoder, self.device).to(self.device)

            # Load pre-trained weights
            if os.path.exists(model_path):
                state_dict = torch.load(model_path, map_location=self.device)
                self.model.load_state_dict(state_dict)
                self.model.eval()

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 50, **kwargs) -> str:
        """Translate text using LSTM model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        self.model.eval()

        if isinstance(text, str):
            tokens = self.src_vocab.numericalize(text)
        else:
            return ""

        tokens = [self.src_vocab.stoi["<SOS>"]] + tokens + [self.src_vocab.stoi["<EOS>"]]
        src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(self.device)

        with torch.no_grad():
            hidden, cell = self.model.encoder(src_tensor)

        trg_indexes = [self.trg_vocab.stoi["<SOS>"]]

        for _ in range(max_length):
            trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(self.device)
            with torch.no_grad():
                output, hidden, cell = self.model.decoder(trg_tensor, hidden, cell)

            pred_token = output.argmax(1).item()
            trg_indexes.append(pred_token)

            if pred_token == self.trg_vocab.stoi["<EOS>"]:
                break

        trg_tokens = [self.trg_vocab.itos[i] for i in trg_indexes]
        return " ".join(trg_tokens[1:-1])

    def translate_batch(self, texts: List[str], **kwargs) -> List[str]:
        """Translate batch of texts using LSTM model."""
        return [self.translate_text(text, **kwargs) for text in texts]


class HelsinkiTranslator(BaseTranslator):
    """Helsinki NLP translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load Helsinki NLP model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, **kwargs) -> str:
        """Translate text using Helsinki model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            outputs = self.model.generate(
                input_ids,
                max_length=max_length,
                do_sample=False
            )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using Helsinki model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        translations = []

        try:
            for text in texts:
                translation = self.translate_text(text, max_length=max_length)
                translations.append(translation)
        except Exception as e:
            pass

        return translations


class NLLBTranslator(BaseTranslator):
    """NLLB (No Language Left Behind) translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load NLLB model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, src_lang: str = "ind_Latn", tgt_lang: str = "eng_Latn", **kwargs) -> str:
        """Translate text using NLLB model. Defaults to Indonesian->English."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            outputs = self.model.generate(
                input_ids,
                forced_bos_token_id=self.tokenizer.convert_tokens_to_ids(tgt_lang),
                max_length=max_length,
                do_sample=False
            )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using NLLB model."""
        return [self.translate_text(text, max_length=max_length, **kwargs) for text in texts]


class MBartTranslator(BaseTranslator):
    """mBART (Multilingual BART) translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load mBART model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, src_lang: str = "id_ID", tgt_lang: str = "en_XX", **kwargs) -> str:
        """Translate text using mBART model. Defaults to Indonesian->English."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            # Handle different language code formats
            if hasattr(self.tokenizer, 'src_lang'):
                self.tokenizer.src_lang = src_lang

            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            # Try different ways to get language token ID
            forced_bos_token_id = None
            if hasattr(self.tokenizer, 'lang_code_to_id') and tgt_lang in self.tokenizer.lang_code_to_id:
                forced_bos_token_id = self.tokenizer.lang_code_to_id[tgt_lang]
            elif hasattr(self.tokenizer, 'convert_tokens_to_ids'):
                try:
                    forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt_lang)
                except:
                    pass

            if forced_bos_token_id:
                outputs = self.model.generate(
                    input_ids,
                    forced_bos_token_id=forced_bos_token_id,
                    max_length=max_length,
                    do_sample=False
                )
            else:
                outputs = self.model.generate(
                    input_ids,
                    max_length=max_length,
                    do_sample=False
                )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using mBART model."""
        return [self.translate_text(text, max_length=max_length, **kwargs) for text in texts]


# Main Translation Class
class Translation:
    """Main Translation class that manages different translation models."""

    def __init__(self, device: Optional[str] = None):
        """
        Initialize the Translation class.

        Args:
            device (str, optional): Device to run models on ('cpu' or 'cuda').
        """
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.translators = {}

    def load_lstm_model(self, model_path: str = "'/content/drive/MyDrive/Model/LSTM2.pt", config: Optional[Dict] = None):
        """Load LSTM translation model."""
        self.translators['lstm'] = LSTMTranslator(self.device, config)
        self.translators['lstm'].load_model(model_path)

    def load_helsinki_model(self, model_name: str = "/content/drive/MyDrive/Model/fine_tuned_helsinki_tourism2"):
        """Load Helsinki NLP translation model."""
        self.translators['helsinki'] = HelsinkiTranslator(self.device)
        self.translators['helsinki'].load_model(model_name)

    def load_nllb_model(self, model_name: str = "/content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2"):
        """Load NLLB translation model."""
        self.translators['nllb'] = NLLBTranslator(self.device)
        self.translators['nllb'].load_model(model_name)

    def load_mbart_model(self, model_name: str = "/content/drive/MyDrive/Model/mbart_fine_tuned_essential2"):
        """Load mBART translation model."""
        self.translators['mbart'] = MBartTranslator(self.device)
        self.translators['mbart'].load_model(model_name)

    def translate_text(self, text: str, model_type: str, **kwargs) -> str:
        """
        Translate text using specified model.

        Args:
            text (str): Text to translate
            model_type (str): Type of model ('lstm', 'helsinki', 'nllb', 'mbart')
            **kwargs: Additional arguments for translation

        Returns:
            str: Translated text
        """
        if model_type not in self.translators:
            raise ValueError(f"Model '{model_type}' not loaded. Available models: {list(self.translators.keys())}")

        return self.translators[model_type].translate_text(text, **kwargs)

    def translate_batch(self, texts: List[str], model_type: str, **kwargs) -> List[str]:
        """
        Translate batch of texts using specified model.

        Args:
            texts (List[str]): List of texts to translate
            model_type (str): Type of model ('lstm', 'helsinki', 'nllb', 'mbart')
            **kwargs: Additional arguments for translation

        Returns:
            List[str]: List of translated texts
        """
        if model_type not in self.translators:
            raise ValueError(f"Model '{model_type}' not loaded. Available models: {list(self.translators.keys())}")

        return self.translators[model_type].translate_batch(texts, **kwargs)

    def get_available_models(self) -> List[str]:
        """Get list of loaded models."""
        return list(self.translators.keys())

    def get_model_info(self, model_type: str = None) -> Dict:
        """
        Get information about loaded models.

        Args:
            model_type (str, optional): Specific model to get info for. If None, returns all models info.

        Returns:
            dict: Model information
        """
        if model_type:
            if model_type not in self.translators:
                raise ValueError(f"Model '{model_type}' not loaded.")
            return self.translators[model_type].get_model_info()
        else:
            return {model_type: translator.get_model_info()
                    for model_type, translator in self.translators.items()}

    def get_translator(self, model_type: str):
        """
        Get specific translator instance.

        Args:
            model_type (str): Type of model ('lstm', 'helsinki', 'nllb', 'mbart')

        Returns:
            BaseTranslator: The translator instance
        """
        if model_type not in self.translators:
            raise ValueError(f"Model '{model_type}' not loaded. Available models: {list(self.translators.keys())}")
        return self.translators[model_type]

In [5]:
import os
import torch
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    PegasusTokenizer,
    PegasusForConditionalGeneration,
)
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sentence_splitter import SentenceSplitter
import nltk
import pandas as pd

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)


class Summarization:
    def __init__(self):
        """Initialize the Summarization class with model configurations."""
        self.MAX_INPUT_LENGTH = 512
        self.MAX_TARGET_LENGTH = 128
        self.loaded_models = {}

        # Determine the base directory for models
        if os.path.exists('/content/drive'):
            # Assuming 'Tubes' is the project root in Google Drive
            base_dir_for_models = "/content/drive/MyDrive/Tubes/Summarization"
            self.models_dir = os.path.join(base_dir_for_models, "models")
        else:
            # Fallback for local execution where __file__ might not be defined.
            # Assumes 'models' directory is at the project root, relative to current working directory.
            self.models_dir = os.path.join(os.getcwd(), "models")

        # Model paths
        self.MODEL_PATHS = {
            "BART_FULL": os.path.join(self.models_dir, "BART_FULL"),
            "BART_SAMPLED": os.path.join(self.models_dir, "BART_SAMPLED"),
            "PEGASUS_FULL": os.path.join(self.models_dir, "PEGASUS_FULL"),
            "PEGASUS_SAMPLED": os.path.join(self.models_dir, "PEGASUS_SAMPLED"),
        }

        # Base model checkpoints for zero-shot
        self.BASE_MODELS = {
            "BART": "facebook/bart-base",
            "PEGASUS": "google/pegasus-large",
        }

    def extract_first_sentence(self, text):
        """Extract the first sentence from text as a simple baseline summary."""
        if pd.isna(text) or not text.strip():
            return ""
        sentences = nltk.sent_tokenize(text)
        return sentences[0].strip() if sentences else ""

    def summarize_textrank(self, text, sentence_count=2):
        """
        Generate extractive summary using TextRank algorithm.

        Args:
            text (str): Input text to summarize
            sentence_count (int): Number of sentences in summary

        Returns:
            str: Extractive summary
        """
        if pd.isna(text) or not text.strip():
            return self.extract_first_sentence(text)

        try:
            parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
            summarizer = TextRankSummarizer()
            splitter = SentenceSplitter(language='en')
            sentences = splitter.split(text)
            num_sentences = min(len(sentences), sentence_count)
            summary = summarizer(parser.document, num_sentences)
            return " ".join([str(s) for s in summary])
        except Exception:
            return self.extract_first_sentence(text)

    def load_model(self, model_type, model_name):
        """
        Load a summarization model (BART or PEGASUS, fine-tuned or zero-shot).

        Args:
            model_type (str): 'BART' or 'PEGASUS'
            model_name (str): Specific model variant (e.g., 'BART_FULL', 'BART_SAMPLED')

        Returns:
            tuple: (model, tokenizer)
        """
        cache_key = model_name if model_name else model_type

        if cache_key in self.loaded_models:
            return self.loaded_models[cache_key]

        # Determine model path
        if model_name and model_name in self.MODEL_PATHS:
            model_path = self.MODEL_PATHS[model_name]
            if not os.path.exists(model_path):
                print(f"Warning: Fine-tuned model not found at {model_path}. Using base model instead.")
                model_path = self.BASE_MODELS[model_type]
        else:
            model_path = self.BASE_MODELS[model_type]

        # Load tokenizer and model
        if model_type == "BART":
            tokenizer = BartTokenizer.from_pretrained(self.BASE_MODELS["BART"])
            model = BartForConditionalGeneration.from_pretrained(model_path)
        elif model_type == "PEGASUS":
            tokenizer = PegasusTokenizer.from_pretrained(self.BASE_MODELS["PEGASUS"])
            model = PegasusForConditionalGeneration.from_pretrained(model_path)
        else:
            raise ValueError(f"Unknown model type: {model_type}")

        # Move to GPU if available
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = model.to(device)
        model.eval()

        self.loaded_models[cache_key] = (model, tokenizer)
        return model, tokenizer

    def generate_summary_transformer(self, text, model_type, model_name=None):
        """
        Generate abstractive summary using transformer models.

        Args:
            text (str): Input text to summarize
            model_type (str): 'BART' or 'PEGASUS'
            model_name (str): Specific model variant (optional)

        Returns:
            str: Generated summary
        """
        if pd.isna(text) or not text.strip():
            return ""

        model, tokenizer = self.load_model(model_type, model_name)

        # Tokenize input
        inputs = tokenizer(
            text,
            max_length=self.MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        # Move to device
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate summary
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=self.MAX_TARGET_LENGTH,
                min_length=10,
                length_penalty=2.0,
                num_beams=4,
                early_stopping=True,
            )

        # Decode and return
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return summary

    def summarize(self, text, method='best'):
        """
        Generate summary using specified method.

        Args:
            text (str): Input text to summarize
            method (str): Summarization method
                - 'first_sentence': Extract first sentence
                - 'textrank': TextRank extractive summarization
                - 'bart_zero_shot': BART base model (zero-shot)
                - 'bart_full': BART fine-tuned on full dataset
                - 'bart_sampled': BART fine-tuned on sampled dataset
                - 'pegasus_zero_shot': PEGASUS base model (zero-shot)
                - 'pegasus_full': PEGASUS fine-tuned on full dataset
                - 'pegasus_sampled': PEGASUS fine-tuned on sampled dataset
                - 'best': Use best performing model (BART_SAMPLED)

        Returns:
            str: Generated summary
        """
        if pd.isna(text) or not text.strip():
            return ""

        method = method.lower()

        # Extractive methods
        if method == 'first_sentence':
            return self.extract_first_sentence(text)
        elif method == 'textrank':
            return self.summarize_textrank(text)

        # BART models
        elif method == 'bart_zero_shot':
            return self.generate_summary_transformer(text, 'BART', None)
        elif method == 'bart_full':
            return self.generate_summary_transformer(text, 'BART', 'BART_FULL')
        elif method == 'bart_sampled':
            return self.generate_summary_transformer(text, 'BART', 'BART_SAMPLED')

        # PEGASUS models
        elif method == 'pegasus_zero_shot':
            return self.generate_summary_transformer(text, 'PEGASUS', None)
        elif method == 'pegasus_full':
            return self.generate_summary_transformer(text, 'PEGASUS', 'PEGASUS_FULL')
        elif method == 'pegasus_sampled':
            return self.generate_summary_transformer(text, 'PEGASUS', 'PEGASUS_SAMPLED')

        # Best model (based on experiment results)
        elif method == 'best':
            return self.generate_summary_transformer(text, 'BART', 'BART_SAMPLED')

        else:
            raise ValueError(f"Unknown summarization method: {method}. "
                           f"Available methods: first_sentence, textrank, "
                           f"bart_zero_shot, bart_full, bart_sampled, "
                           f"pegasus_zero_shot, pegasus_full, pegasus_sampled, best")

    def get_available_models(self):
        """
        Get list of available models with their status.

        Returns:
            dict: Model availability status
        """
        available = {
            'extractive': ['first_sentence', 'textrank'],
            'bart': {
                'zero_shot': True,
                'full': os.path.exists(self.MODEL_PATHS.get('BART_FULL', '')),
                'sampled': os.path.exists(self.MODEL_PATHS.get('BART_SAMPLED', ''))
            },
            'pegasus': {
                'zero_shot': True,
                'full': os.path.exists(self.MODEL_PATHS.get('PEGASUS_FULL', '')),
                'sampled': os.path.exists(self.MODEL_PATHS.get('PEGASUS_SAMPLED', ''))
            }
        }
        return available


In [6]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification,
    GPT2Tokenizer, BartTokenizer, XLMRobertaTokenizer
)
import torch
import joblib
import os
import re
from typing import List, Optional, Dict, Any
from abc import ABC, abstractmethod

try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except ImportError:
    TEXTBLOB_AVAILABLE = False
    print("Warning: TextBlob not available. Rule-based ABSA will use fallback methods.")

VALID_ASPECTS = [
    'place', 'spot', 'view', 'price', 'quietness', 'food', 'drink', 'maintenance',
    'staff', 'peacefulness', 'road', 'air', 'water', 'comfort', 'photo', 'music',
    'restroom', 'parking', 'taxi', 'spaciousness', 'light', 'accessibility',
    'cleanliness', 'atmosphere', 'management', 'service', 'facility',
    'location', 'weather', 'crowd', 'safety', 'entrance', 'ticket', 'guide',
    'information', 'wifi', 'internet', 'scenery', 'beauty', 'nature',
    'architecture', 'history', 'culture', 'tradition', 'souvenir', 'shopping',
    'accommodation', 'hotel', 'restaurant', 'cafe', 'transportation',
    'walking', 'hiking', 'trail', 'path', 'signage', 'direction', 'map',
    'security', 'noise', 'crowdedness', 'queue', 'waiting', 'opening_hours',
    'tour', 'activity', 'entertainment', 'attraction', 'monument', 'museum',
    'garden', 'beach', 'mountain', 'lake', 'river', 'bridge', 'temple',
    'religious', 'spiritual', 'calm', 'serenity', 'landscape'
]

class BaseABSA(ABC):
    def __init__(self, device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.is_loaded = False
        self.valid_aspects = VALID_ASPECTS

    @abstractmethod
    def extract_aspects(self, text: str) -> List[str]:
        pass

    @abstractmethod
    def classify_aspect(self, text: str, real_aspect: str) -> str:
        pass

    @abstractmethod
    def predict_sentiment(self, text: str, aspect: str) -> str:
        pass

    def get_model_info(self) -> dict:
        return {
            "model_type": self.__class__.__name__,
            "device": self.device,
            "is_loaded": self.is_loaded
        }

class RuleBasedABSA(BaseABSA):
    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.is_loaded = True
        self._textblob_ready = self._check_textblob()

    def _check_textblob(self):
        if not TEXTBLOB_AVAILABLE:
            return False

        try:
            test = TextBlob("test")
            _ = test.tags
            _ = test.noun_phrases
            return True
        except Exception as e:
            print(f"[Rule-based] TextBlob not fully configured: {e}")
            print("[Rule-based] Using fallback methods")
            return False

    def extract_aspects(self, text: str) -> List[str]:
        aspects = []

        if self._textblob_ready:
            try:
                blob = TextBlob(text)

                try:
                    for np in blob.noun_phrases:
                        if len(np.split()) <= 3:
                            aspects.append(np.strip())
                except Exception:
                    pass

                try:
                    for word, pos in blob.tags:
                        if pos.startswith('NN') and len(word) > 2:
                            aspects.append(word.lower())
                except Exception:
                    pass

                if not aspects:
                    aspects = self._fallback_extract_aspects(text)

            except Exception as e:
                print(f"[Rule-based] TextBlob extraction failed: {e}")
                aspects = self._fallback_extract_aspects(text)
        else:
            aspects = self._fallback_extract_aspects(text)

        seen = set()
        unique_aspects = []
        for asp in aspects:
            if asp and asp not in seen:
                seen.add(asp)
                unique_aspects.append(asp)

        return unique_aspects if unique_aspects else ['place']

    def _fallback_extract_aspects(self, text: str) -> List[str]:
        aspect_keywords = [
            'place', 'location', 'spot', 'view', 'scenery', 'beach', 'mountain',
            'food', 'restaurant', 'cafe', 'price', 'staff', 'service', 'facility',
            'hotel', 'accommodation', 'room', 'parking', 'toilet', 'restroom',
            'entrance', 'ticket', 'guide', 'tour', 'activity', 'atmosphere',
            'cleanliness', 'maintenance', 'crowd', 'weather', 'water', 'air',
            'museum', 'temple', 'garden', 'nature', 'culture', 'history'
        ]

        text_lower = text.lower()
        aspects = []
        for keyword in aspect_keywords:
            if keyword in text_lower:
                aspects.append(keyword)

        common_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
                       'for', 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are',
                       'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do',
                       'does', 'did', 'will', 'would', 'should', 'could', 'can', 'may'}

        words = re.findall(r'\b[a-zA-Z]+\b', text)
        for word in words:
            word_lower = word.lower()
            if len(word_lower) > 3 and word_lower not in common_words:
                aspects.append(word_lower)

        return aspects if aspects else ['place']

    def classify_aspect(self, text: str, real_aspect: str) -> str:
        if not real_aspect:
            return 'place'

        real_asp_lower = real_aspect.lower()

        if real_asp_lower in [va.lower() for va in self.valid_aspects]:
            idx = [va.lower() for va in self.valid_aspects].index(real_asp_lower)
            return self.valid_aspects[idx]

        for valid_asp in self.valid_aspects:
            if valid_asp.lower() in real_asp_lower or real_asp_lower in valid_asp.lower():
                return valid_asp

        mapping = {
            'building': 'architecture',
            'site': 'place',
            'area': 'location',
            'cost': 'price',
            'fee': 'price',
            'people': 'crowd',
            'workers': 'staff',
            'employees': 'staff',
            'meal': 'food',
            'dish': 'food',
            'beverage': 'drink',
            'bathroom': 'restroom',
            'wc': 'restroom',
            'car park': 'parking',
            'lot': 'parking',
        }

        for key, value in mapping.items():
            if key in real_asp_lower:
                return value

        return 'place'

    def predict_sentiment(self, text: str, aspect: str) -> str:
        if self._textblob_ready:
            try:
                sentence_with_aspect = f"{text} [ASPECT: {aspect}]"
                blob = TextBlob(sentence_with_aspect)
                polarity = blob.sentiment.polarity

                if polarity > 0.1:
                    return 'positive'
                elif polarity < -0.1:
                    return 'negative'
                else:
                    return 'neutral'
            except Exception as e:
                print(f"[Rule-based] TextBlob sentiment failed: {e}")
                return self._fallback_predict_sentiment(text)
        else:
            return self._fallback_predict_sentiment(text)

    def _fallback_predict_sentiment(self, text: str) -> str:
        text_lower = text.lower()

        positive_words = [
            'good', 'great', 'excellent', 'amazing', 'wonderful', 'beautiful',
            'nice', 'perfect', 'love', 'best', 'awesome', 'fantastic', 'clean',
            'comfortable', 'friendly', 'helpful', 'delicious', 'stunning',
            'brilliant', 'superb', 'magnificent', 'incredible', 'outstanding',
            'lovely', 'pretty', 'enjoyable', 'pleasant', 'impressive'
        ]

        negative_words = [
            'bad', 'poor', 'terrible', 'awful', 'horrible', 'dirty', 'rude',
            'expensive', 'crowded', 'disappointing', 'worst', 'ugly', 'broken',
            'smelly', 'noisy', 'unfriendly', 'unhelpful', 'overpriced', 'disgusting',
            'unpleasant', 'mediocre', 'inferior', 'lacking', 'inadequate', 'subpar'
        ]

        pos_count = sum(1 for word in positive_words if word in text_lower)
        neg_count = sum(1 for word in negative_words if word in text_lower)

        if pos_count > neg_count:
            return 'positive'
        elif neg_count > pos_count:
            return 'negative'
        else:
            return 'neutral'

class TransformerABSA(BaseABSA):
    def __init__(self, model_base_path: str, device: Optional[str] = None, supports_task1: bool = True):
        super().__init__(device)
        self.model_base_path = model_base_path
        self.models = {}
        self.tokenizers = {}
        self.supports_task1 = supports_task1
        self.rule_based_fallback = RuleBasedABSA(device) if not supports_task1 else None

        # Label mappings
        self.aspect_label_map = {asp: idx for idx, asp in enumerate(VALID_ASPECTS)}
        self.reverse_aspect_label_map = {idx: asp for idx, asp in enumerate(VALID_ASPECTS)}
        self.sentiment_label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
        self.reverse_sentiment_label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
        self.real_aspect_label_map = {'O': 0, 'B-ASPECT': 1, 'I-ASPECT': 2}
        self.reverse_real_aspect_label_map = {0: 'O', 1: 'B-ASPECT', 2: 'I-ASPECT'}

    def load_model(self, task: str):
        if task in self.models:
            return

        model_path = os.path.join(self.model_base_path, task)

        if not os.path.exists(model_path):
            if task == 'task1' and not self.supports_task1:
                return
            raise FileNotFoundError(f"Model path does not exist: {model_path}")

        try:
            if task == 'task1':
                model = AutoModelForTokenClassification.from_pretrained(
                    model_path,
                    local_files_only=True
                )
            else:
                model = AutoModelForSequenceClassification.from_pretrained(
                    model_path,
                    local_files_only=True
                )

            tokenizer = AutoTokenizer.from_pretrained(
                model_path,
                local_files_only=True
            )

            model.to(self.device)
            model.eval()

            self.models[task] = model
            self.tokenizers[task] = tokenizer
            self.is_loaded = True
        except Exception as e:
            raise RuntimeError(f"Failed to load model from {model_path}: {e}")

    def extract_aspects(self, text: str) -> List[str]:
        if not self.supports_task1:
            print(f"[Task 1] Model doesn't support Task 1, using rule-based fallback")
            return self.rule_based_fallback.extract_aspects(text)

        if 'task1' not in self.models:
            self.load_model('task1')

        model = self.models['task1']
        tokenizer = self.tokenizers['task1']

        try:
            tokens = text.split()
            inputs = tokenizer(
                tokens,
                is_split_into_words=True,
                return_tensors="pt",
                truncation=True,
                max_length=128
            )

            word_ids = inputs.word_ids()
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                predictions = torch.argmax(outputs.logits, dim=2)

            predicted_labels = predictions[0].tolist()

            aspects = []
            current_aspect = []

            for word_idx, label_id in zip(word_ids, predicted_labels):
                if word_idx is None:
                    continue

                label = self.reverse_real_aspect_label_map.get(label_id, 'O')

                if label == 'B-ASPECT':
                    if current_aspect:
                        aspects.append(' '.join(current_aspect))
                    current_aspect = [tokens[word_idx]]
                elif label == 'I-ASPECT' and current_aspect:
                    current_aspect.append(tokens[word_idx])
                elif label == 'O' and current_aspect:
                    aspects.append(' '.join(current_aspect))
                    current_aspect = []

            if current_aspect:
                aspects.append(' '.join(current_aspect))

            return aspects if aspects else []
        except Exception as e:
            print(f"[Task 1] Extraction failed: {e}")
            return []

    def classify_aspect(self, text: str, real_aspect: str) -> str:
        if 'task2' not in self.models:
            self.load_model('task2')

        model = self.models['task2']
        tokenizer = self.tokenizers['task2']

        try:
            input_text = f"{text} [SEP] {real_aspect}"
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=128,
                padding='max_length'
            )

            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                predictions = torch.argmax(outputs.logits, dim=1)

            pred_label = predictions[0].item()
            return self.reverse_aspect_label_map.get(pred_label, 'place')
        except Exception as e:
            print(f"[Task 2] Classification failed: {e}")
            return 'place'

    def predict_sentiment(self, text: str, aspect: str) -> str:
        if 'task3' not in self.models:
            self.load_model('task3')

        model = self.models['task3']
        tokenizer = self.tokenizers['task3']

        try:
            input_text = f"{text} [SEP] {aspect}"
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=128,
                padding='max_length'
            )

            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                predictions = torch.argmax(outputs.logits, dim=1)

            pred_label = predictions[0].item()
            return self.reverse_sentiment_label_map.get(pred_label, 'neutral')
        except Exception as e:
            print(f"[Task 3] Sentiment prediction failed: {e}")
            return 'neutral'

class TfidfLRABSA(BaseABSA):
    def __init__(self, models_dir: str, device: Optional[str] = None):
        super().__init__(device)
        self.models_dir = models_dir
        self.task2_model = None
        self.task2_vectorizer = None
        self.task3_model = None
        self.task3_vectorizer = None
        self.rule_based = RuleBasedABSA(device)

    def load_models(self):
        try:
            self.task2_model = joblib.load(os.path.join(self.models_dir, "tfidf_lr_task2_model.pkl"))
            self.task2_vectorizer = joblib.load(os.path.join(self.models_dir, "tfidf_lr_task2_vectorizer.pkl"))
            self.task3_model = joblib.load(os.path.join(self.models_dir, "tfidf_lr_task3_model.pkl"))
            self.task3_vectorizer = joblib.load(os.path.join(self.models_dir, "tfidf_lr_task3_vectorizer.pkl"))
            self.is_loaded = True
        except Exception as e:
            raise RuntimeError(f"Error loading TF-IDF LR models: {e}")

    def extract_aspects(self, text: str) -> List[str]:
        return self.rule_based.extract_aspects(text)

    def classify_aspect(self, text: str, real_aspect: str) -> str:
        if not self.is_loaded:
            self.load_models()

        try:
            input_text = f"{text} [REAL_ASPECT: {real_aspect}]"
            X = self.task2_vectorizer.transform([input_text])
            prediction = self.task2_model.predict(X)[0]
            return prediction
        except Exception as e:
            print(f"[TF-IDF Task 2] Classification failed: {e}")
            return 'place'

    def predict_sentiment(self, text: str, aspect: str) -> str:
        if not self.is_loaded:
            self.load_models()

        try:
            input_text = f"{text} [ASPECT: {aspect}]"
            X = self.task3_vectorizer.transform([input_text])
            prediction = self.task3_model.predict(X)[0]
            return prediction
        except Exception as e:
            print(f"[TF-IDF Task 3] Sentiment prediction failed: {e}")
            return 'neutral'

# Main ABSA Class
class ABSA:
    def __init__(self, device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.models = {}
        self.base_path = self._get_base_path()

    def _get_base_path(self) -> str:
        if os.path.exists('/content/drive'):
            return "/content/drive/MyDrive/Tubes/ABSA/models"
        else:
            base_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
            return os.path.join(base_dir, "models", "absa")

    def load_bert_model(self, model_name: str = "comp2_mono-bert-base"):
        model_path = os.path.join(self.base_path, model_name)
        self.models['bert'] = TransformerABSA(model_path, self.device)
        print(f"✓ BERT model loaded from: {model_path}")

    def load_gpt2_model(self, model_name: str = "comp2_decoder-only-gpt2"):
        model_path = os.path.join(self.base_path, model_name)
        self.models['gpt2'] = TransformerABSA(model_path, self.device, supports_task1=False)
        print(f"✓ GPT2 model loaded from: {model_path}")

    def load_bart_model(self, model_name: str = "comp2_encoder-decoder-bart"):
        model_path = os.path.join(self.base_path, model_name)
        self.models['bart'] = TransformerABSA(model_path, self.device, supports_task1=False)
        print(f"✓ BART model loaded from: {model_path}")

    def load_xlm_roberta_model(self, model_name: str = "comp2_multi-xlm-roberta-base"):
        model_path = os.path.join(self.base_path, model_name)
        self.models['xlm_roberta'] = TransformerABSA(model_path, self.device, supports_task1=False)
        print(f"✓ XLM-RoBERTa model loaded from: {model_path}")

    def load_tfidf_lr_model(self):
        self.models['tfidf_lr'] = TfidfLRABSA(self.base_path, self.device)
        self.models['tfidf_lr'].load_models()
        print("✓ TF-IDF + LR models loaded")

    def load_rule_based(self):
        self.models['rule_based'] = RuleBasedABSA(self.device)
        print("✓ Rule-based model loaded")

    def analyze(self, text: str, task1_model: str = 'bert', task2_model: str = 'bert',
                task3_model: str = 'bert') -> List[Dict[str, str]]:
        results = []

        try:
            # Task 1: Extract aspects
            if task1_model not in self.models:
                raise ValueError(f"Model '{task1_model}' not loaded for Task 1. Available models: {self.get_available_models()}")

            print(f"[Task 1] Extracting aspects using: {task1_model}")
            real_aspects = self.models[task1_model].extract_aspects(text)

            if not real_aspects:
                print("[Task 1] No aspects found in text")
                return []

            print(f"[Task 1] Found aspects: {real_aspects}")

            for real_aspect in real_aspects:
                # Task 2: Classify aspect
                if task2_model not in self.models:
                    raise ValueError(f"Model '{task2_model}' not loaded for Task 2. Available models: {self.get_available_models()}")

                print(f"[Task 2] Classifying aspect '{real_aspect}' using: {task2_model}")
                aspect_category = self.models[task2_model].classify_aspect(text, real_aspect)
                print(f"[Task 2] Classified to category: {aspect_category}")

                # Task 3: Predict sentiment using PREDICTED category (not original aspect)
                if task3_model not in self.models:
                    raise ValueError(f"Model '{task3_model}' not loaded for Task 3. Available models: {self.get_available_models()}")

                print(f"[Task 3] Predicting sentiment using: {task3_model}")
                sentiment = self.models[task3_model].predict_sentiment(text, aspect_category)
                print(f"[Task 3] Sentiment: {sentiment}")

                results.append({
                    'real_aspect': real_aspect,
                    'aspect_category': aspect_category,
                    'sentiment': sentiment
                })

        except Exception as e:
            print(f"[ABSA Error] Analysis failed: {e}")
            import traceback
            traceback.print_exc()
            return []

        return results

    def get_available_models(self) -> List[str]:
        return list(self.models.keys())

    def get_model_info(self, model_type: str = None) -> Dict:
        if model_type:
            if model_type not in self.models:
                raise ValueError(f"Model '{model_type}' not loaded.")
            return self.models[model_type].get_model_info()
        else:
            return {model_type: model.get_model_info()
                    for model_type, model in self.models.items()}

In [7]:
import time
import sys
import os
import random

LIST_MODEL_TRANSLATE = [
    'helsinki',
    'nllb',
    'mbart',
    'lstm'
]


LIST_MODEL_ABSA_TASK1 = [
    'bert',
    'rule_based',
    'best'
]

LIST_MODEL_ABSA_TASK2 = [
    'bert',
    'gpt2',
    'bart',
    'xlm_roberta',
    'rule_based',
    'best'
]

LIST_MODEL_ABSA_TASK3 = [
    'bert',
    'gpt2',
    'bart',
    'xlm_roberta',
    'rule_based',
    'best'
]

LIST_MODEL_SUMMARIZATION = [
    'first_sentence',
    'textrank',
    'bart_zero_shot',
    'bart_full',
    'bart_sampled',
    'pegasus_zero_shot',
    'pegasus_full',
    'pegasus_sampled',
    'best'
]

def clear_screen():
    """Clear the terminal screen."""
    os.system('cls' if os.name == 'nt' else 'clear')

def print_banner():
    """Display a stylish banner."""
    banner = """
    ╔══════════════════════════════════════════════════════════╗
    ║                                                          ║
    ║      ██╗███╗   ██╗██████╗  ██████╗ ████████╗██████╗      ║
    ║      ██║████╗  ██║██╔══██╗██╔═══██╗╚══██╔══╝██╔══██╗     ║
    ║      ██║██╔██╗ ██║██║  ██║██║   ██║   ██║   ██████╔╝     ║
    ║      ██║██║╚██╗██║██║  ██║██║   ██║   ██║   ██╔══██╗     ║
    ║      ██║██║ ╚████║██████╔╝╚██████╔╝   ██║   ██║  ██║     ║
    ║      ╚═╝╚═╝  ╚═══╝╚═════╝  ╚═════╝    ╚═╝   ╚═╝  ╚═╝     ║
    ║                                                          ║
    ║      ████████╗██████╗ ██╗██████╗ ███████╗██╗             ║
    ║      ╚══██╔══╝██╔══██╗██║██╔══██╗██╔════╝██║             ║
    ║         ██║   ██████╔╝██║██████╔╝███████╗██║             ║
    ║         ██║   ██╔══██╗██║██╔═══╝ ╚════██║██║             ║
    ║         ██║   ██║  ██║██║██║     ███████║██║             ║
    ║         ╚═╝   ╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝             ║
    ║                                                          ║
    ║          IndoTripSight NLP Pipeline System               ║
    ║                                                          ║
    ╚══════════════════════════════════════════════════════════╝
    """
    print(banner)

def animate_typing(text, delay=0.03):
    """Animate text appearing character by character."""
    for char in text:
        sys.stdout.write(char)
        sys.stdout.flush()
        time.sleep(delay)
    print()

def show_progress_bar(message, duration=2, width=50):
    """Display an animated progress bar."""
    print(f"\n{message}")
    print("[" + " " * width + "]", end="")
    sys.stdout.flush()

    for i in range(width + 1):
        progress = i / width
        filled = int(width * progress)
        bar = "█" * filled + "░" * (width - filled)
        percentage = int(progress * 100)

        sys.stdout.write(f"\r[{bar}] {percentage}%")
        sys.stdout.flush()
        time.sleep(duration / width)

    print(f"\n[COMPLETE] {message} - Complete!")

def animate_spinner(message, duration=2):
    """Display an animated spinner."""
    spinner_chars = ["⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏"]
    end_time = time.time() + duration

    while time.time() < end_time:
        for char in spinner_chars:
            if time.time() >= end_time:
                break
            sys.stdout.write(f'\r{message} {char}')
            sys.stdout.flush()
            time.sleep(0.1)

    sys.stdout.write(f'\r{message} [DONE]\n')
    sys.stdout.flush()

def show_loading_dots(message, duration=2, dot_count=3):
    """Display loading dots animation."""
    end_time = time.time() + duration

    while time.time() < end_time:
        for i in range(dot_count + 1):
            if time.time() >= end_time:
                break
            dots = "." * i + " " * (dot_count - i)
            sys.stdout.write(f'\r{message}{dots}')
            sys.stdout.flush()
            time.sleep(0.3)

    sys.stdout.write(f'\r{message}{"." * dot_count} [DONE]\n')
    sys.stdout.flush()

def animate_glitch_text(text, iterations=5):
    """Create a glitch effect for text."""
    glitch_chars = "!@#$%^&*()_+-=[]{}|;:,.<>?~`"
    original = text
    for _ in range(iterations):
        glitched = ''.join(random.choice(glitch_chars) if random.random() < 0.3 else c for c in original)
        sys.stdout.write(f'\r{glitched}')
        sys.stdout.flush()
        time.sleep(0.1)
    sys.stdout.write(f'\r{original}\n')
    sys.stdout.flush()

def show_wave_animation(message="Welcome", cycles=3):
    """Display a wave animation."""
    wave_frames = [
        "     ┌───────────────────────────┐",
        "    ╔═══╗                        ",
        "   ╔═══╝  IndoTripSight NLP      ",
        "  ╔═══╗   Pipeline System        ",
        " ╔═══╝                           ",
        "╔═══╗                            "
    ]

    for cycle in range(cycles):
        for frame in wave_frames:
            clear_screen()
            print_banner()
            print("\n")
            print(" " * 20 + frame)
            print("\n" + " " * 25 + message)
            time.sleep(0.2)

    clear_screen()
    print_banner()

def celebrate_completion():
    """Display celebration animation."""
    messages = [
        "*", "#", "=", "+", "-", "~"
    ]

    for _ in range(3):
        celebration = " ".join(random.choice(messages) for _ in range(15))
        print("\r" + celebration, end="")
        time.sleep(0.3)

    print("\n")

def get_user_choice(options_list, prompt):
    """Get user choice by number from a list of options."""
    while True:
        try:
            choice = int(input(prompt).strip())
            if 1 <= choice <= len(options_list):
                return options_list[choice - 1]
            else:
                print(f"Please enter a number between 1 and {len(options_list)}")
        except ValueError:
            print("Please enter a valid number")


class IndoTripSight:
    def __init__(self):
        self.reviews = []
        self.translated_text = None
        self.absa_processor = ABSA()
        self.translator = Translation()
        self.summarizer = Summarization()
        self.translation_models_loaded = {}
        self.absa_models_loaded = {}

    def load_absa_models(self, models_to_load=None):
        """Load specified ABSA models."""
        if models_to_load is None:
            models_to_load = ['bert', 'rule_based']

        print("Loading ABSA models...")

        if 'bert' in models_to_load:
            try:
                self.absa_processor.load_bert_model()
                self.absa_models_loaded['bert'] = True
            except Exception as e:
                print(f"✗ BERT model failed: {e}")
                self.absa_models_loaded['bert'] = False

        if 'gpt2' in models_to_load:
            try:
                self.absa_processor.load_gpt2_model()
                self.absa_models_loaded['gpt2'] = True
            except Exception as e:
                print(f"✗ GPT2 model failed: {e}")
                self.absa_models_loaded['gpt2'] = False

        if 'bart' in models_to_load:
            try:
                self.absa_processor.load_bart_model()
                self.absa_models_loaded['bart'] = True
            except Exception as e:
                print(f"✗ BART model failed: {e}")
                self.absa_models_loaded['bart'] = False

        if 'xlm_roberta' in models_to_load:
            try:
                self.absa_processor.load_xlm_roberta_model()
                self.absa_models_loaded['xlm_roberta'] = True
            except Exception as e:
                print(f"✗ XLM-RoBERTa model failed: {e}")
                self.absa_models_loaded['xlm_roberta'] = False

        if 'tfidf_lr' in models_to_load:
            try:
                self.absa_processor.load_tfidf_lr_model()
                self.absa_models_loaded['tfidf_lr'] = True
            except Exception as e:
                print(f"✗ TF-IDF LR model failed: {e}")
                self.absa_models_loaded['tfidf_lr'] = False

        if 'rule_based' in models_to_load:
            try:
                self.absa_processor.load_rule_based()
                self.absa_models_loaded['rule_based'] = True
            except Exception as e:
                print(f"✗ Rule-based model failed: {e}")
                self.absa_models_loaded['rule_based'] = False

        print(f"ABSA models loaded: {list(k for k, v in self.absa_models_loaded.items() if v)}")

    def load_translation_models(self, models_to_load=None):
        """Load specified translation models or all available models."""
        if models_to_load is None:
            models_to_load = ['helsinki', 'nllb', 'mbart', 'lstm']

        print("Loading translation models...")

        if 'helsinki' in models_to_load:
            try:
                self.translator.load_helsinki_model("/content/drive/MyDrive/Tubes/Translation/Model/fine_tuned_helsinki_tourism2")
                self.translation_models_loaded['helsinki'] = True
                print("✓ Helsinki model loaded")
            except Exception as e:
                print(f"✗ Helsinki model failed: {e}")
                self.translation_models_loaded['helsinki'] = False

        if 'nllb' in models_to_load:
            try:
                self.translator.load_nllb_model("/content/drive/MyDrive/Tubes/Translation/Model/fine_tuned_nllb_tourism_essential2")
                self.translation_models_loaded['nllb'] = True
                print("✓ NLLB model loaded")
            except Exception as e:
                print(f"✗ NLLB model failed: {e}")
                self.translation_models_loaded['nllb'] = False

        if 'mbart' in models_to_load:
            try:
                self.translator.load_mbart_model("/content/drive/MyDrive/Tubes/Translation/Model/mbart_fine_tuned_essential2")
                self.translation_models_loaded['mbart'] = True
                print("✓ mBART model loaded")
            except Exception as e:
                print(f"✗ mBART model failed: {e}")
                self.translation_models_loaded['mbart'] = False

        if 'lstm' in models_to_load:
            try:
                self.translator.load_lstm_model("/content/drive/MyDrive/Tubes/Translation/Model/LSTM2.pt")
                self.translation_models_loaded['lstm'] = True
                print("✓ LSTM model loaded")
            except Exception as e:
                print(f"✗ LSTM model failed: {e}")
                self.translation_models_loaded['lstm'] = False

        print(f"Translation models loaded: {list(k for k, v in self.translation_models_loaded.items() if v)}")

    def add_review(self, review):
        self.reviews.append(review)

    def translate_review(self, model='nllb'):
        if not self.reviews:
            return "No reviews to translate"
        review_text = self.reviews[-1]

        try:
            translator_type = model

            # Check if model is loaded
            if translator_type not in self.translation_models_loaded or not self.translation_models_loaded[translator_type]:
                self.translated_text = review_text
                return f"[{model.upper()}] Model not loaded. Using original text: {review_text}"

            # Perform actual translation
            if translator_type == 'lstm':
                translated = self.translator.translate_text(review_text, 'lstm')
            elif translator_type == 'helsinki':
                translated = self.translator.translate_text(review_text, 'helsinki')
            elif translator_type == 'nllb':
                translated = self.translator.translate_text(review_text, 'nllb',
                                                          src_lang="ind_Latn", tgt_lang="eng_Latn")
            elif translator_type == 'mbart':
                translated = self.translator.translate_text(review_text, 'mbart',
                                                          src_lang="id_ID", tgt_lang="en_XX")
            else:
                translated = review_text

            # Store translated text for ABSA
            self.translated_text = translated if translated else review_text

            return f"[{model.upper()}] Translated: {self.translated_text}"

        except Exception as e:
            self.translated_text = review_text
            return f"[{model.upper()}] Translation failed: {str(e)}. Using original: {review_text}"

    def get_translation_model_status(self):
        """Get status of loaded translation models."""
        return {
            'loaded_models': [k for k, v in self.translation_models_loaded.items() if v],
            'failed_models': [k for k, v in self.translation_models_loaded.items() if not v],
            'total_loaded': sum(self.translation_models_loaded.values()),
            'available_models': self.translator.get_available_models() if hasattr(self.translator, 'get_available_models') else []
        }

    def absa(self, task1_model='best', task2_model='best', task3_model='best'):
        """Perform ABSA with separate model selection for each task

        Args:
            task1_model: Model for aspect extraction
            task2_model: Model for aspect classification
            task3_model: Model for sentiment classification
        """
        if self.translated_text is None:
             self.translate_review()

        translated_review = self.translated_text

        # Default 'best' to 'bert'
        if task1_model == 'best':
            task1_model = 'bert'
        if task2_model == 'best':
            task2_model = 'bert'
        if task3_model == 'best':
            task3_model = 'bert'

        try:
            # Perform ABSA analysis using the new module
            results = self.absa_processor.analyze(
                translated_review,
                task1_model=task1_model,
                task2_model=task2_model,
                task3_model=task3_model
            )

            # Format results
            if results:
                formatted_results = f"\n[ABSA] Task1:{task1_model} | Task2:{task2_model} | Task3:{task3_model}\n"
                for result in results:
                    formatted_results += f"  • {result['real_aspect']} ({result['aspect_category']}) → {result['sentiment']}\n"
                return formatted_results.strip()
            else:
                return f"[ABSA] No aspects found in: {translated_review}"

        except Exception as e:
            import traceback
            traceback.print_exc()
            return f"[Error] ABSA analysis failed: {str(e)}"

    def summarization(self, model='best'):
        """Generate summary of the translated review using specified model.

        Args:
            model (str): Summarization method to use

        Returns:
            str: Generated summary
        """
        # Use the already translated text if available, otherwise translate first
        if self.translated_text is None:
            self.translate_review('nllb')  # Use default translation model

        translated_review = self.translated_text

        # Generate summary using the Summarization module
        try:
            summary = self.summarizer.summarize(translated_review, method=model)
            return summary
        except Exception as e:
            print(f"Error in summarization: {e}")
            return f"Error generating summary: {str(e)}"

In [8]:
clear_screen()

# Show banner once at startup
print_banner()

indo_trip_sight = IndoTripSight()

# Initialize translation models
print("\nInitializing Translation Models...")
models_to_load = ['nllb', 'helsinki', 'mbart', 'lstm']
indo_trip_sight.load_translation_models(models_to_load)

# Show model status
status = indo_trip_sight.get_translation_model_status()
print(f"\n✓ Translation models ready: {len(status['loaded_models'])}/{len(models_to_load)}")
if status['loaded_models']:
    print(f"   Loaded: {', '.join(status['loaded_models'])}")
if status['failed_models']:
    print(f"   Failed: {', '.join(status['failed_models'])}")

# Initialize ABSA models
print("\nInitializing ABSA Models...")
absa_models_to_load = ['bert', 'rule_based']
indo_trip_sight.load_absa_models(absa_models_to_load)

print(f"\n✓ ABSA models ready: {len([k for k, v in indo_trip_sight.absa_models_loaded.items() if v])}/{len(absa_models_to_load)}")

print("\nIndoTripSight NLP Pipeline System Ready")

# Get user input for text
print("\n")
text = input("Enter the review text: ").strip()
if text:
    indo_trip_sight.add_review(text)
    print("Review added to pipeline")
else:
    print("[ERROR] No text entered. Exiting...")
    sys.exit(1)

# Translation Model Selection
print("\nAvailable Translation Models:")
for i, model in enumerate(LIST_MODEL_TRANSLATE, 1):
    print(f"   {i}. {model}")

translate_model = get_user_choice(
    LIST_MODEL_TRANSLATE,
    f"\nChoose translation model (1-{len(LIST_MODEL_TRANSLATE)}): "
)

# ABSA Model Selection - Separate for each task
print("\nABSA Model Selection (3 Tasks)")

print("\n[Task 1: Aspect Extraction]")
print("Available models:")
for i, model in enumerate(LIST_MODEL_ABSA_TASK1, 1):
    print(f"   {i}. {model}")

absa_task1_model = get_user_choice(
    LIST_MODEL_ABSA_TASK1,
    f"\nChoose Task 1 model (1-{len(LIST_MODEL_ABSA_TASK1)}): "
)

print("\n[Task 2: Aspect Category Classification]")
print("Available models:")
for i, model in enumerate(LIST_MODEL_ABSA_TASK2, 1):
    print(f"   {i}. {model}")

absa_task2_model = get_user_choice(
    LIST_MODEL_ABSA_TASK2,
    f"\nChoose Task 2 model (1-{len(LIST_MODEL_ABSA_TASK2)}): "
)

print("\n[Task 3: Sentiment Classification]")
print("Available models:")
for i, model in enumerate(LIST_MODEL_ABSA_TASK3, 1):
    print(f"   {i}. {model}")

absa_task3_model = get_user_choice(
    LIST_MODEL_ABSA_TASK3,
    f"\nChoose Task 3 model (1-{len(LIST_MODEL_ABSA_TASK3)}): "
)

# Summarization Model Selection
print("\nAvailable Summarization Models:")
for i, model in enumerate(LIST_MODEL_SUMMARIZATION, 1):
    print(f"   {i}. {model}")

summarization_model = get_user_choice(
    LIST_MODEL_SUMMARIZATION,
    f"\nChoose summarization model (1-{len(LIST_MODEL_SUMMARIZATION)}): "
)

print("\nConfiguration Summary:")
print(f"   Review: {text}")
print(f"   Translation: {translate_model}")
print(f"   ABSA Task 1 (Aspect Extraction): {absa_task1_model}")
print(f"   ABSA Task 2 (Aspect Classification): {absa_task2_model}")
print(f"   ABSA Task 3 (Sentiment): {absa_task3_model}")
print(f"   Summarization: {summarization_model}")

# Show selected model info
selected_translator_model = translate_model

if selected_translator_model in status['loaded_models']:
    print(f"\n✓ Using loaded {selected_translator_model.upper()} model for translation")
else:
    print(f"\n⚠ {selected_translator_model.upper()} model not loaded, will use fallback")

# Process translation
print("\nTranslating text...")
translated = indo_trip_sight.translate_review(translate_model)
print(f"Translation Result: {translated}")

# Process ABSA
print("\nAnalyzing aspects and sentiment...")
absa_result = indo_trip_sight.absa(
    task1_model=absa_task1_model,
    task2_model=absa_task2_model,
    task3_model=absa_task3_model
)
print(f"ABSA Result:\n{absa_result}")

# Process summarization
print("\nGenerating summary...")
summary = indo_trip_sight.summarization(summarization_model)
print(f"Summary Result: {summary}")

print("\nProcessing complete! Thank you for using IndoTripSight NLP Pipeline!")


    ╔══════════════════════════════════════════════════════════╗
    ║                                                          ║
    ║      ██╗███╗   ██╗██████╗  ██████╗ ████████╗██████╗      ║
    ║      ██║████╗  ██║██╔══██╗██╔═══██╗╚══██╔══╝██╔══██╗     ║
    ║      ██║██╔██╗ ██║██║  ██║██║   ██║   ██║   ██████╔╝     ║
    ║      ██║██║╚██╗██║██║  ██║██║   ██║   ██║   ██╔══██╗     ║
    ║      ██║██║ ╚████║██████╔╝╚██████╔╝   ██║   ██║  ██║     ║
    ║      ╚═╝╚═╝  ╚═══╝╚═════╝  ╚═════╝    ╚═╝   ╚═╝  ╚═╝     ║
    ║                                                          ║
    ║      ████████╗██████╗ ██╗██████╗ ███████╗██╗             ║
    ║      ╚══██╔══╝██╔══██╗██║██╔══██╗██╔════╝██║             ║
    ║         ██║   ██████╔╝██║██████╔╝███████╗██║             ║
    ║         ██║   ██╔══██╗██║██╔═══╝ ╚════██║██║             ║
    ║         ██║   ██║  ██║██║██║     ███████║██║             ║
    ║         ╚═╝   ╚═╝  ╚═╝╚═╝╚═╝     ╚══════╝╚═╝             ║
    ║                   

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✓ Helsinki model loaded
✓ NLLB model loaded
✓ mBART model loaded


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


✓ LSTM model loaded
Translation models loaded: ['helsinki', 'nllb', 'mbart', 'lstm']

✓ Translation models ready: 4/4
   Loaded: helsinki, nllb, mbart, lstm

Initializing ABSA Models...
Loading ABSA models...
✓ BERT model loaded from: /content/drive/MyDrive/Tubes/ABSA/models/comp2_mono-bert-base
[Rule-based] TextBlob not fully configured: 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.

[Rule-based] Using fallback methods
✓ Rule-based model loaded
ABSA models loaded: ['bert', 'rule_based']

✓ ABSA models ready: 2/2

IndoTripSight NLP Pipeline System Ready


Enter the review text: Gunung putri tempat yg indah untuk menikmati view kota dari tempat ketinggian. Jarak dari tempat parkir mobil ke puncak kira2 satu kil

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Summary Result: Mount Princess offers beautiful city views from its altitude. Access involves a one-kilometer walk from the parking lot to the summit, with an elevation of 100. Facilities include a coffee stall, an indomie, and a bathroom near the car park.

Processing complete! Thank you for using IndoTripSight NLP Pipeline!
